# 03 - ANNOTATION DES CORPUS

Etape **Annotation** : on transforme chaque livre nettoyé en un tableau où chaque ligne est un token enrichi de ses propriétés linguistiques (lemme, partie du discours, etc.). C'est ce tableau qui alimente l'exploration visuelle puis les modules suivants (TF-IDF, similarite, resume MMR).

**Choix techniques** :

- Modèle `fr_core_news_sm` (12 Mo, rapide). Suffisant sur du francais standard ; on changera pour `_md` si l'exploration révèle des erreurs systématiques.
- Pipeline `tagger + parser + lemmatizer` actifs, **`ner` désactivée**. La NER de spaCy est entrainée sur du news moderne et fait beaucoup d'erreurs sur du XIXe siecle. On garde le parser parce qu'il fournit la segmentation en phrases (`doc.sents`).

La logique vit dans `pipeline.annotation`. Elle découpe le texte en chunks (frontières sur double saut de ligne pour ne pas casser au milieu d'une phrase), envoie via `nlp.pipe()` pour le traitement en batch, et concatène les tokens en un DataFrame unique. Les `sent_id` et `token_id` sont continus d'un chunk au suivant.


## Setup

In [1]:
import sys
from pathlib import Path

# Se déplacer dans le dossier du projet (NE LANCER Q'UNE FOIS)
%cd ../
%ls

/home/gau/projets/nlp/projet_nlp_app
README.md  figures/  notebooks/      pipeline/         scripts/
app/       models/   pg_catalog.csv  requirements.txt


In [2]:
import time

from pipeline import storage
from pipeline.annotation import annoter, _get_nlp
from pipeline.config import ANNOTATE_PARAMS, PREFIXES

nlp = _get_nlp()
print(f"Modele : {nlp.meta['name']} v{nlp.meta['version']}")
print(f"Langue : {nlp.meta['lang']}")
print(f"Pipeline actif : {nlp.pipe_names}")

Modele : core_news_sm v3.8.0
Langue : fr
Pipeline actif : ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer']


## Vérification rapide sur une phrase

Avant de lancer sur le vrai corpus, on s'assure que le modèle se comporte normalement sur une phrase de référence.


In [3]:
phrase = "Le docteur Pascal cherchait un dossier dans l'armoire de chene massif. Clotilde dessinait, assise pres de la fenetre."
df_test = annoter(phrase)
df_test

,sent_id,token_id,text,lemma,pos,tag,is_punct,is_stop,is_alpha
0,0,0,Le,le,DET,DET,False,True,True
1,0,1,docteur,docteur,NOUN,NOUN,False,False,True
2,0,2,Pascal,pascal,ADJ,ADJ,False,False,True
3,0,3,cherchait,chercher,VERB,VERB,False,False,True
4,0,4,un,un,DET,DET,False,True,True
5,0,5,dossier,dossier,NOUN,NOUN,False,False,True
6,0,6,dans,dans,ADP,ADP,False,True,True
7,0,7,l',le,DET,DET,False,True,False
8,0,8,armoire,armoire,NOUN,NOUN,False,False,True
9,0,9,de,de,ADP,ADP,False,True,True


## Lister les livres nettoyés

In [4]:
livres_propres = storage.list_objects(PREFIXES["clean"], suffix=".txt")
print(f"{len(livres_propres)} livres à annoter")
for o in livres_propres[:5]:
    print(" -", o["Key"])

26 livres à annoter
 - clean/adventure/dumas_alexandre/pg17990_le_comte_de_monte_cristo_tome_ii.txt
 - clean/adventure/raspe_rudolf_erich/pg50398_aventures_de_baron_de_munchausen.txt
 - clean/biography/fusil_louise/pg26720_souvenirs_dune_actrice_23.txt
 - clean/biography/marmont_auguste_frederic_louis_viesse_de_duc_de_raguse/pg30013_memoires_du_marechal_marmont_duc_de_raguse_39.txt
 - clean/biography/rouquette_louis_frederic/pg70801_lepopee_blanche.txt


## Test sur un livre complet

On annote un livre entier pour avoir un ordre de grandeur du temps de traitement et de la taille de sortie. C'est aussi l'occasion de regarder a quoi ressemblent les premières lignes du DataFrame produit.


In [5]:
cle_test = livres_propres[0]["Key"]
texte = storage.get_text(cle_test)

t0 = time.time()
df = annoter(texte)
duree = time.time() - t0

print(f"Livre   : {cle_test}")
print(f"Duree   : {duree:.1f} s")
print(f"Tokens  : {len(df):,}")
print(f"Phrases : {df['sent_id'].nunique():,}")
print(f"Memoire : {df.memory_usage(deep=True).sum() / 1024**2:.1f} Mo")
print()
print("--- 15 premiers tokens ---")
print(df.head(15))
print()
print("--- Distribution des POS ---")
print(df["pos"].value_counts())
print()
print("--- Top 10 lemmes hors ponctuation et mots-outils ---")
lemmes = df.loc[df["is_alpha"] & ~df["is_stop"], "lemma"].str.lower()
print(lemmes.value_counts().head(10))

Livre   : clean/adventure/dumas_alexandre/pg17990_le_comte_de_monte_cristo_tome_ii.txt
Duree   : 11.0 s
Tokens  : 149,859
Phrases : 6,075
Memoire : 5.3 Mo

--- 15 premiers tokens ---
    sent_id  token_id       text      lemma    pos    tag  is_punct  is_stop  \
0         0         0         LE         le    DET    DET     False     True   
1         0         1      COMTE      comte   NOUN   NOUN     False    False   
2         0         2         DE         de    ADP    ADP     False     True   
3         0         3      MONTE      MONTE  PROPN  PROPN     False    False   
4         0         4          -          -  PROPN  PROPN      True    False   
5         0         5     CRISTO     CRISTO  PROPN  PROPN     False    False   
6         1         6  Alexandre  Alexandre  PROPN  PROPN     False    False   
7         1         7      Dumas      Dumas  PROPN  PROPN     False    False   
8         2         8       Tome       tome   NOUN   NOUN     False    False   
9         2      

## Annoter tous les livres et uploader sous `annotations/`

Miroir de l'arborescence : `clean/genre/auteur/livre.txt` devient `annotations/genre/auteur/livre.parquet`.


In [6]:
for obj in livres_propres:
    cle = obj["Key"]
    texte = storage.get_text(cle)

    t0 = time.time()
    df = annoter(texte)
    duree = time.time() - t0

    sous_cle = cle[len(PREFIXES["clean"]):]
    if sous_cle.endswith(".txt"):
        sous_cle = sous_cle[:-4] + ".parquet"
    nouvelle_cle = PREFIXES["annotations"] + sous_cle

    storage.put_parquet(
        nouvelle_cle, df,
        metadata={
            "source":       "annotations",
            "original_key": cle,
            "n_tokens":     str(len(df)),
            "n_phrases":    str(df["sent_id"].nunique()),
        },
    )

    print(f"{cle}: {len(df):>7,} tokens, {df['sent_id'].nunique():>5,} phrases, {duree:>5.1f}s")

clean/adventure/dumas_alexandre/pg17990_le_comte_de_monte_cristo_tome_ii.txt: 149,859 tokens, 6,075 phrases,  10.0s
clean/adventure/raspe_rudolf_erich/pg50398_aventures_de_baron_de_munchausen.txt:  32,128 tokens, 1,136 phrases,   2.9s
clean/biography/fusil_louise/pg26720_souvenirs_dune_actrice_23.txt:  65,809 tokens, 2,874 phrases,   4.9s
clean/biography/marmont_auguste_frederic_louis_viesse_de_duc_de_raguse/pg30013_memoires_du_marechal_marmont_duc_de_raguse_39.txt: 133,226 tokens, 4,576 phrases,   9.1s
clean/biography/rouquette_louis_frederic/pg70801_lepopee_blanche.txt:  52,879 tokens, 2,782 phrases,   4.3s
clean/biography/savary_anne_jean_marie_rene_duc_de_rovigo/pg20895_memoires_du_duc_de_rovigo_pour_servir_a_lhistoire_de_lempereur_napoleon_tome_2.txt: 114,808 tokens, 4,159 phrases,   7.3s
clean/biography/stendhal/pg30977_la_vie_de_rossini_tome_i.txt:  80,273 tokens, 3,248 phrases,   5.5s
clean/historical_fiction/dumas_alexandre/pg17989_le_comte_de_monte_cristo_tome_i.txt: 153,836 

## Tests de cohérence sur un livre annoté

Onze tests rapides pour s'assurer que l'annotation est saine. A exécuter après `Annoter tous les livres`.


In [1]:
# Charge un livre annoté
cle_test = livres_propres[0]["Key"]
sous_cle = cle_test[len(PREFIXES["clean"]):]
cle_annot = PREFIXES["annotations"] + sous_cle.replace(".txt", ".parquet")
df = storage.get_parquet(cle_annot)

print(f"Livre : {cle_annot}")
print(f"Tokens : {len(df):,}")
print(f"Phrases : {df['sent_id'].nunique():,}")
print(f"Vocabulaire (lemmes) : {df['lemma'].nunique():,}")
print()

# === TEST 1 : structure ===
attendu = {"sent_id", "token_id", "text", "lemma", "pos", "tag",
           "is_punct", "is_stop", "is_alpha"}
manquant = attendu - set(df.columns)
print(f"[1] Colonnes complètes : {'OK' if not manquant else 'MANQUE ' + str(manquant)}")

# === TEST 2 : pas de NaN dans les colonnes critiques ===
nans = df[["text", "lemma", "pos"]].isna().sum().sum()
print(f"[2] Pas de NaN dans text/lemma/pos : {'OK' if nans == 0 else f'{nans} NaN'}")

# === TEST 3 : token_id continu ===
ids = df["token_id"].values
continu = (ids[1:] - ids[:-1] == 1).all()
print(f"[3] token_id strictement croissant : {'OK' if continu else 'CASSE'}")

# === TEST 4 : sent_id croissant (peut stagner, ne doit pas reculer) ===
sids = df["sent_id"].values
recul = ((sids[1:] - sids[:-1]) < 0).any()
print(f"[4] sent_id ne recule pas : {'OK' if not recul else 'RECULE'}")

# === TEST 5 : longueur moyenne de phrase plausible ===
tok_par_phrase = df.groupby("sent_id").size()
moy = tok_par_phrase.mean()
print(f"[5] Moyenne tokens/phrase : {moy:.1f} ({'OK' if 10 < moy < 50 else 'SUSPECT'})")

# === TEST 6 : phrases trop longues ? ===
top_long = tok_par_phrase.nlargest(3)
print(f"[6] Top 3 phrases les plus longues : {top_long.tolist()}")
print(f"    {'OK' if top_long.max() < 500 else 'SUSPECT'}")

# === TEST 7 : distribution POS dominee par les bonnes categories ===
pos_counts = df["pos"].value_counts(normalize=True)
print(f"[7] Top 5 POS :")
for p, pct in pos_counts.head(5).items():
    print(f"    {p:8s} {pct:.1%}")

# === TEST 8 : lemmatisation - quelques verbes connus ===
print(f"[8] Vérification de lemmes attendus :")
for forme, lemme_attendu in [("est", "etre"), ("avait", "avoir"),
                              ("disait", "dire"), ("vint", "venir")]:
    sub = df[df["text"].str.lower() == forme]
    if len(sub) > 0:
        lemmes_trouves = sub["lemma"].str.lower().value_counts()
        top_lemme = lemmes_trouves.index[0]
        ok = lemme_attendu in lemmes_trouves.index[:3].tolist()
        print(f"    '{forme}' -> {top_lemme} (attendu: {lemme_attendu}) {'OK' if ok else 'à vérifier'}")
    else:
        print(f"    '{forme}' : absent du livre")

# === TEST 9 : flags coherents ===
incoherent = ((df["is_punct"]) & (df["is_alpha"])).sum()
print(f"[9] is_punct et is_alpha mutuellement exclusifs : "
      f"{'OK' if incoherent == 0 else f'{incoherent} cas incohérents'}")

# === TEST 10 : apercu d'une phrase reconstruite ===
print(f"\n[10] Phrase #100 reconstruite :")
phrase = df[df["sent_id"] == 100]
print("    " + " ".join(phrase["text"].tolist()))

NameError: name 'livres_propres' is not defined